In [2]:
message = "Han, Mitakuyap!"
print(message)

Han, Mitakuyap!


In [3]:

# ============================================================
# SA 19 — DISSERTATION FIGURES & STATISTICS GALLERY
# Environmental Data Cube for Oglala Lakota
#
# READ-ONLY:
# - Reads saved outputs from Notebook 09
# - Does not rerun satellite analysis
# - Does not change NetCDF or research files
# - Does not require new installations
# ============================================================

from pathlib import Path
import json
import base64
import html
import re

import ipywidgets as widgets
from IPython.display import (
    display, HTML, Image, clear_output
)

# ------------------------------------------------------------
# 1. LOCATE DISSERTATION NOTEBOOK
# ------------------------------------------------------------

project = Path(
    "/Users/elishawakinyanzi/Documents/"
    "ElishaWakinyanZi-WORK/datacube_2026/"
    "ate_makoce_agcube"
)

notebook_path = (
    project
    / "Notebooks"
    / "09_dissertation_data_cube_validation.ipynb"
)

if not notebook_path.is_file():
    raise FileNotFoundError(
        "Could not find Notebook 09 at:\n"
        + str(notebook_path)
    )

with notebook_path.open(
    "r", encoding="utf-8"
) as f:
    notebook = json.load(f)

cells = notebook.get("cells", [])

print("Notebook loaded:", notebook_path.name)
print("Total cells:", len(cells))


# ------------------------------------------------------------
# 2. COLLECT SAVED FIGURES AND RESULTS
# ------------------------------------------------------------

records = []

def join_text(value):
    if isinstance(value, list):
        return "".join(value)
    return str(value or "")

for cell_number, cell in enumerate(cells, start=1):

    if cell.get("cell_type") != "code":
        continue

    outputs = cell.get("outputs", [])

    if not outputs:
        continue

    source = join_text(cell.get("source", ""))

    # Use nearby markdown headings for context.
    previous_headings = []

    for prior in cells[
        max(0, cell_number - 9):cell_number - 1
    ]:
        if prior.get("cell_type") == "markdown":
            text = join_text(prior.get("source", ""))

            for line in text.splitlines():
                if line.strip().startswith("#"):
                    previous_headings.append(
                        line.strip().lstrip("#").strip()
                    )

    heading = (
        previous_headings[-1]
        if previous_headings
        else "Unlabeled analysis"
    )

    has_image = False
    has_table = False
    has_text = False

    for output in outputs:
        data = output.get("data", {})

        if "image/png" in data or "image/jpeg" in data:
            has_image = True

        if "text/html" in data:
            has_table = True

        if (
            "text/plain" in data
            or output.get("output_type") == "stream"
        ):
            has_text = True

    if not (has_image or has_table or has_text):
        continue

    search_text = (
        heading + "\n" + source + "\n"
        + "\n".join(
            join_text(
                output.get("text", "")
            )
            + "\n"
            + join_text(
                output.get("data", {}).get(
                    "text/plain", ""
                )
            )
            for output in outputs
        )
    ).lower()

    records.append({
        "cell": cell_number,
        "heading": heading,
        "source": source,
        "outputs": outputs,
        "image": has_image,
        "table": has_table,
        "text": has_text,
        "search": search_text
    })

print("Cells with saved outputs:", len(records))
print(
    "Cells containing figures:",
    sum(r["image"] for r in records)
)
print(
    "Cells containing HTML results:",
    sum(r["table"] for r in records)
)


# ------------------------------------------------------------
# 3. DEFINE RESULT CATEGORIES
# ------------------------------------------------------------

categories = {
    "All results": None,

    "Figures and maps": [
        "figure", "plot", "map", "raster",
        "imshow", "scatter", "histogram",
        "heatmap", "chart"
    ],

    "Box plots": [
        "boxplot", "box plot", "boxen",
        "quartile", "interquartile"
    ],

    "Correlations": [
        "correlation", "pearson",
        "spearman", "corrcoef",
        "pearsonr", "spearmanr",
        "corr("
    ],

    "Spatial statistics": [
        "moran", "spatial autocorrelation",
        "spatial correlation", "geary"
    ],

    "Descriptive statistics": [
        "describe(", "mean", "median",
        "standard deviation", "std",
        "variance", "percentile",
        "quantile"
    ],

    "Validation and QA": [
        "validation", "quality",
        "qa", "accuracy",
        "reliability", "agreement",
        "completeness"
    ],

    "Sensitivity and uncertainty": [
        "sensitivity", "bootstrap",
        "confidence interval",
        "uncertainty", "threshold",
        "seasonal window"
    ],

    "NDVI and NDMI": [
        "ndvi", "ndmi"
    ],

    "MODIS and EVI": [
        "modis", "evi", "mod13"
    ],

    "Climate results": [
        "precip", "temperature",
        "tmax", "climate",
        "drought"
    ]
}


# ------------------------------------------------------------
# 4. CREATE INTERACTIVE CONTROLS
# ------------------------------------------------------------

category = widgets.Dropdown(
    options=list(categories),
    value="All results",
    description="Category:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="390px")
)

search = widgets.Text(
    placeholder=(
        "Search: Pearson, boxplot, Moran, "
        "bootstrap, NDVI..."
    ),
    description="Search:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="500px")
)

results_selector = widgets.Select(
    options=[],
    rows=12,
    description="Results:",
    style={"description_width": "90px"},
    layout=widgets.Layout(
        width="850px",
        height="270px"
    )
)

show_code = widgets.Checkbox(
    value=False,
    description="Show original analysis code",
    indent=False
)

result_count = widgets.HTML()
viewer = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #E5D8D8",
        padding="14px",
        width="850px"
    )
)


# ------------------------------------------------------------
# 5. FILTER RESULTS
# ------------------------------------------------------------

def matches_filter(record):

    selected = category.value
    terms = categories[selected]

    if selected == "Figures and maps":
        category_match = record["image"]

    elif selected == "All results":
        category_match = True

    else:
        category_match = any(
            term in record["search"]
            for term in terms
        )

    search_term = search.value.strip().lower()

    search_match = (
        not search_term
        or search_term in record["search"]
    )

    return category_match and search_match


def update_list(change=None):

    filtered = [
        r for r in records
        if matches_filter(r)
    ]

    options = [
        (
            f'Cell {r["cell"]:03d} — '
            f'{r["heading"][:75]}',
            r["cell"]
        )
        for r in filtered
    ]

    results_selector.options = options

    result_count.value = (
        '<div style="padding:10px;'
        'color:#742631;font-weight:600;">'
        f'{len(filtered)} matching result cells'
        '</div>'
    )

    if options:
        results_selector.value = options[0][1]
    else:
        with viewer:
            clear_output(wait=True)
            print("No saved outputs match this filter.")


# ------------------------------------------------------------
# 6. DISPLAY SAVED OUTPUTS
# ------------------------------------------------------------

def show_result(change=None):

    selected_cell = results_selector.value

    if selected_cell is None:
        return

    record = next(
        (
            r for r in records
            if r["cell"] == selected_cell
        ),
        None
    )

    if record is None:
        return

    with viewer:
        clear_output(wait=True)

        display(HTML(
            '<div style="background:#742631;'
            'color:white;padding:16px;'
            'border-radius:10px;">'
            '<h3 style="color:white;margin:0;">'
            f'Cell {record["cell"]}: '
            f'{html.escape(record["heading"])}'
            '</h3></div>'
        ))

        if show_code.value:
            display(HTML(
                '<h4>Original analysis code</h4>'
                '<pre style="white-space:pre-wrap;'
                'background:#F7F2F1;padding:12px;">'
                + html.escape(record["source"][:12000])
                + '</pre>'
            ))

        for output in record["outputs"]:

            data = output.get("data", {})

            # PNG figures
            if "image/png" in data:

                encoded = join_text(data["image/png"])

                try:
                    display(Image(
                        data=base64.b64decode(encoded)
                    ))
                except Exception as error:
                    print("Could not display image:", error)

            # JPEG figures
            elif "image/jpeg" in data:

                encoded = join_text(data["image/jpeg"])

                try:
                    display(Image(
                        data=base64.b64decode(encoded),
                        format="jpeg"
                    ))
                except Exception as error:
                    print("Could not display image:", error)

            # HTML tables and formatted outputs
            elif "text/html" in data:

                display(HTML(
                    join_text(data["text/html"])
                ))

            # Plain text statistics
            elif "text/plain" in data:

                text = join_text(data["text/plain"])

                display(HTML(
                    '<pre style="white-space:pre-wrap;'
                    'font-size:13px;">'
                    + html.escape(text[:30000])
                    + '</pre>'
                ))

            # Printed output
            elif output.get("output_type") == "stream":

                text = join_text(
                    output.get("text", "")
                )

                display(HTML(
                    '<pre style="white-space:pre-wrap;'
                    'font-size:13px;">'
                    + html.escape(text[:30000])
                    + '</pre>'
                ))


# ------------------------------------------------------------
# 7. CONNECT INTERACTIVE CONTROLS
# ------------------------------------------------------------

category.observe(
    update_list,
    names="value"
)

search.observe(
    update_list,
    names="value"
)

results_selector.observe(
    show_result,
    names="value"
)

show_code.observe(
    show_result,
    names="value"
)


# ------------------------------------------------------------
# 8. DISPLAY DISSERTATION RESULTS GALLERY
# ------------------------------------------------------------

header = widgets.HTML("""
<div style="
    background:linear-gradient(125deg,#681C27,#9B303D);
    color:white;
    border-radius:16px;
    padding:26px;
    margin-bottom:15px;
    font-family:Arial,sans-serif;
">
    <div style="
        font-size:12px;
        letter-spacing:1.2px;
        color:#F3D7D2;
    ">
        ENVIRONMENTAL DATA CUBE FOR OGLALA LAKOTA
    </div>

    <h2 style="
        color:white;
        margin:10px 0;
    ">
        Dissertation Figures & Statistical Results
    </h2>

    <p style="color:#F9E8E4;">
        SA 19 — Research Visualization and Validation Gallery
    </p>

    <p style="
        font-size:13px;
        color:#F3D7D2;
    ">
        Original saved outputs from
        09_dissertation_data_cube_validation.ipynb
    </p>
</div>
""")

gallery = widgets.VBox([
    header,
    category,
    search,
    result_count,
    results_selector,
    show_code,
    viewer
])

update_list()
display(gallery)


Notebook loaded: 09_dissertation_data_cube_validation.ipynb
Total cells: 483
Cells with saved outputs: 384
Cells containing figures: 80
Cells containing HTML results: 228


In [4]:

# ============================================================
# SA 19B — DISSERTATION RESULTS BATCH SEARCH & EXPORT
# Environmental Data Cube for Oglala Lakota
#
# Searches Notebook 09 by section, code, and saved output.
# Exports figures, tables, and statistical results together.
#
# READS Notebook 09 — DOES NOT RERUN ANALYSES.
# Writes only selected exports to a new results folder.
# ============================================================

from pathlib import Path
from datetime import datetime
import json
import re
import csv
import io
import zipfile
import html

import ipywidgets as widgets
from IPython.display import display, HTML, FileLink, clear_output


# ------------------------------------------------------------
# 1. LOAD NOTEBOOK 09
# ------------------------------------------------------------

project = Path(
    "/Users/elishawakinyanzi/Documents/"
    "ElishaWakinyanZi-WORK/datacube_2026/"
    "ate_makoce_agcube"
)

source_path = (
    project / "Notebooks"
    / "09_dissertation_data_cube_validation.ipynb"
)

if not source_path.is_file():
    raise FileNotFoundError(
        f"Notebook 09 not found: {source_path}"
    )

with source_path.open("r", encoding="utf-8") as f:
    source_notebook = json.load(f)

source_cells = source_notebook["cells"]


def as_text(value):
    if isinstance(value, list):
        return "".join(value)
    return str(value or "")


# ------------------------------------------------------------
# 2. IDENTIFY ANALYSIS SECTIONS
# ------------------------------------------------------------

# Detect SA section identifiers in Markdown headings.
# Examples: SA 15C, SA 16.17, SA 18.4, SA 18.5

section_pattern = re.compile(
    r"\bSA\s*[-:]?\s*(\d+(?:\.\d+)?[A-Za-z]?)\b",
    flags=re.IGNORECASE
)

catalog = []
current_section = "Unassigned"
current_heading = "Unlabeled analysis"

for number, cell in enumerate(source_cells, start=1):

    source = as_text(cell.get("source", ""))

    if cell.get("cell_type") == "markdown":

        headings = [
            line.lstrip("#").strip()
            for line in source.splitlines()
            if line.lstrip().startswith("#")
        ]

        if headings:
            current_heading = headings[0]

        # Prefer an SA identifier in the heading.
        heading_matches = [
            section_pattern.search(h)
            for h in headings
        ]

        section_match = next(
            (m for m in heading_matches if m),
            None
        )

        if section_match:
            current_section = (
                "SA " + section_match.group(1).upper()
            )

        continue

    if cell.get("cell_type") != "code":
        continue

    outputs = cell.get("outputs", [])

    if not outputs:
        continue

    # Also detect section labels in code comments.
    first_lines = "\n".join(source.splitlines()[:12])

    code_match = section_pattern.search(first_lines)

    if code_match:
        current_section = (
            "SA " + code_match.group(1).upper()
        )

    # Search saved text outputs too.
    output_text = []

    image_count = 0
    html_count = 0

    for output in outputs:

        data = output.get("data", {})

        output_text.append(
            as_text(output.get("text", ""))
        )

        output_text.append(
            as_text(data.get("text/plain", ""))
        )

        if (
            "image/png" in data
            or "image/jpeg" in data
            or "image/svg+xml" in data
        ):
            image_count += 1

        if "text/html" in data:
            html_count += 1

    searchable = "\n".join([
        current_section,
        current_heading,
        source,
        *output_text
    ]).lower()

    catalog.append({
        "cell": number,
        "section": current_section,
        "heading": current_heading,
        "source": source,
        "outputs": outputs,
        "search": searchable,
        "images": image_count,
        "html_tables": html_count
    })

print("Notebook 09 loaded.")
print("Code cells with saved outputs:", len(catalog))
print(
    "Image outputs:",
    sum(r["images"] for r in catalog)
)


# ------------------------------------------------------------
# 3. INTERACTIVE SEARCH CONTROLS
# ------------------------------------------------------------

section_options = [
    ("SA 15C — Climate and statistical models", "SA 15C"),
    ("SA 16.17 — Seasonal-window validation", "SA 16.17"),
    ("SA 18.4 — Environmental assessment", "SA 18.4"),
    ("SA 18.5 — Sensitivity and reliability", "SA 18.5")
]

section_select = widgets.SelectMultiple(
    options=section_options,
    value=tuple(value for _, value in section_options),
    description="Sections:",
    rows=4,
    style={"description_width": "85px"},
    layout=widgets.Layout(width="700px")
)

keyword_search = widgets.Text(
    value="",
    placeholder=(
        "Optional: correlation, Pearson, boxplot, "
        "bootstrap, Moran, NDVI..."
    ),
    description="Search:",
    style={"description_width": "85px"},
    layout=widgets.Layout(width="700px")
)

search_mode = widgets.Dropdown(
    options=[
        ("Any matching keyword", "any"),
        ("All keywords", "all")
    ],
    value="any",
    description="Match:",
    style={"description_width": "85px"},
    layout=widgets.Layout(width="330px")
)

content_type = widgets.Dropdown(
    options=[
        ("All saved results", "all"),
        ("Figures only", "figures"),
        ("Tables only", "tables"),
        ("Text/statistics only", "text")
    ],
    value="all",
    description="Include:",
    style={"description_width": "85px"},
    layout=widgets.Layout(width="330px")
)

include_code = widgets.Checkbox(
    value=False,
    description="Include original analysis code in export",
    indent=False
)

search_button = widgets.Button(
    description="Find Results",
    button_style="info",
    icon="search",
    layout=widgets.Layout(width="170px")
)

export_button = widgets.Button(
    description="Export Matching Results",
    button_style="danger",
    icon="download",
    layout=widgets.Layout(width="235px")
)

summary_output = widgets.Output()
export_output = widgets.Output()

matched_records = []


# ------------------------------------------------------------
# 4. SEARCH NOTEBOOK
# ------------------------------------------------------------

def search_records():

    selected = set(section_select.value)

    # Split terms by commas; spaces within terms are preserved.
    raw = keyword_search.value.strip()

    terms = [
        term.strip().lower()
        for term in raw.split(",")
        if term.strip()
    ]

    results = []

    for record in catalog:

        if selected:
            if record["section"] not in selected:
                continue

        if terms:
            if search_mode.value == "all":
                if not all(
                    term in record["search"]
                    for term in terms
                ):
                    continue
            else:
                if not any(
                    term in record["search"]
                    for term in terms
                ):
                    continue

        if content_type.value == "figures":
            if record["images"] == 0:
                continue

        elif content_type.value == "tables":
            if record["html_tables"] == 0:
                continue

        elif content_type.value == "text":
            has_text = any(
                output.get("output_type") == "stream"
                or "text/plain" in output.get("data", {})
                for output in record["outputs"]
            )
            if not has_text:
                continue

        results.append(record)

    return results


def run_search(_=None):

    global matched_records

    matched_records = search_records()

    with summary_output:

        clear_output(wait=True)

        print("DISSERTATION RESULTS SEARCH")
        print("=" * 45)
        print("Matching cells:", len(matched_records))
        print(
            "Saved image outputs:",
            sum(r["images"] for r in matched_records)
        )
        print(
            "Saved HTML outputs:",
            sum(r["html_tables"] for r in matched_records)
        )

        print("\nMATCHING ANALYSES\n")

        for record in matched_records[:100]:

            print(
                f'Cell {record["cell"]:03d} | '
                f'{record["section"]} | '
                f'{record["heading"][:65]} | '
                f'{record["images"]} images, '
                f'{record["html_tables"]} HTML outputs'
            )

        if len(matched_records) > 100:
            print(
                f"\n... and "
                f"{len(matched_records) - 100} more cells."
            )

        if not matched_records:
            print(
                "\nNo matches. Try clearing the keyword "
                "search or changing the section selection."
            )


# ------------------------------------------------------------
# 5. EXPORT FIGURES, TABLES, AND STATISTICS
# ------------------------------------------------------------

def export_results(_):

    # Always refresh the search before exporting.
    results = search_records()

    with export_output:

        clear_output(wait=True)

        if not results:
            print("No matching results to export.")
            return

        export_dir = (
            project / "results"
            / "SA19_dissertation_exports"
        )

        export_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        timestamp = datetime.now().strftime(
            "%Y%m%d_%H%M%S"
        )

        zip_path = (
            export_dir
            / f"dissertation_results_{timestamp}.zip"
        )

        index_rows = []

        with zipfile.ZipFile(
            zip_path,
            "w",
            compression=zipfile.ZIP_DEFLATED
        ) as archive:

            for record in results:

                cell_number = record["cell"]

                section_name = re.sub(
                    r"[^A-Za-z0-9._-]+",
                    "_",
                    record["section"]
                )

                folder = (
                    f"{section_name}/"
                    f"cell_{cell_number:03d}"
                )

                if include_code.value:

                    archive.writestr(
                        f"{folder}/analysis_code.py",
                        record["source"]
                    )

                for output_number, output in enumerate(
                    record["outputs"], start=1
                ):

                    data = output.get("data", {})

                    filename = None
                    file_content = None

                    # Save figures in original format.
                    if "image/png" in data:

                        import base64

                        filename = (
                            f"figure_{output_number:02d}.png"
                        )

                        file_content = base64.b64decode(
                            as_text(data["image/png"])
                        )

                    elif "image/jpeg" in data:

                        import base64

                        filename = (
                            f"figure_{output_number:02d}.jpg"
                        )

                        file_content = base64.b64decode(
                            as_text(data["image/jpeg"])
                        )

                    elif "image/svg+xml" in data:

                        filename = (
                            f"figure_{output_number:02d}.svg"
                        )

                        file_content = as_text(
                            data["image/svg+xml"]
                        )

                    elif "text/html" in data:

                        filename = (
                            f"table_or_output_{output_number:02d}.html"
                        )

                        file_content = as_text(
                            data["text/html"]
                        )

                    elif "text/plain" in data:

                        filename = (
                            f"statistics_{output_number:02d}.txt"
                        )

                        file_content = as_text(
                            data["text/plain"]
                        )

                    elif output.get("output_type") == "stream":

                        filename = (
                            f"printed_results_{output_number:02d}.txt"
                        )

                        file_content = as_text(
                            output.get("text", "")
                        )

                    if filename is None:
                        continue

                    # Respect the selected content type.
                    extension = Path(filename).suffix.lower()

                    if content_type.value == "figures":
                        if extension not in (
                            ".png", ".jpg", ".svg"
                        ):
                            continue

                    elif content_type.value == "tables":
                        if extension != ".html":
                            continue

                    elif content_type.value == "text":
                        if extension != ".txt":
                            continue

                    archive_name = f"{folder}/{filename}"

                    archive.writestr(
                        archive_name,
                        file_content
                    )

                    index_rows.append({
                        "section": record["section"],
                        "notebook_cell": cell_number,
                        "heading": record["heading"],
                        "filename": archive_name,
                        "type": extension,
                        "source_notebook": source_path.name
                    })

            # Add a CSV index for dissertation traceability.
            csv_buffer = io.StringIO()

            writer = csv.DictWriter(
                csv_buffer,
                fieldnames=[
                    "section",
                    "notebook_cell",
                    "heading",
                    "filename",
                    "type",
                    "source_notebook"
                ]
            )

            writer.writeheader()
            writer.writerows(index_rows)

            archive.writestr(
                "dissertation_results_index.csv",
                csv_buffer.getvalue()
            )

            archive.writestr(
                "README.txt",
                (
                    "Environmental Data Cube for Oglala Lakota\n"
                    "SA 19B Dissertation Results Export\n\n"
                    f"Source: {source_path.name}\n"
                    f"Exported cells: {len(results)}\n"
                    f"Exported files: {len(index_rows)}\n\n"
                    "These are saved notebook outputs.\n"
                    "They have not been recalculated.\n"
                    "Consult original code and source data "
                    "before publication.\n"
                )
            )

        print("EXPORT COMPLETE")
        print("=" * 45)
        print("Matching cells:", len(results))
        print("Files exported:", len(index_rows))
        print("\nSaved to:")
        print(zip_path)

        display(FileLink(
            str(zip_path),
            result_html_prefix=(
                "<b>Download selected dissertation results:</b> "
            )
        ))


search_button.on_click(run_search)
export_button.on_click(export_results)


# ------------------------------------------------------------
# 6. DISPLAY BATCH SEARCH AND EXPORTER
# ------------------------------------------------------------

header = widgets.HTML("""
<div style="
    background:linear-gradient(125deg,#681C27,#9B303D);
    color:white;
    border-radius:16px;
    padding:24px;
    margin-bottom:12px;
">
    <h2 style="color:white;margin:0 0 8px;">
        Dissertation Results Batch Exporter
    </h2>

    <p style="color:#F9E8E4;">
        Search SA 15C, SA 16.17, SA 18.4, and SA 18.5
        and download their saved figures and statistics.
    </p>

    <small>
        Environmental Data Cube for Oglala Lakota
        — SA 19B
    </small>
</div>
""")

exporter = widgets.VBox([
    header,
    section_select,
    keyword_search,
    widgets.HBox([
        search_mode,
        content_type
    ]),
    include_code,
    widgets.HBox([
        search_button,
        export_button
    ]),
    summary_output,
    export_output
])

display(exporter)

# Show the initial matching inventory.
run_search()


Notebook 09 loaded.
Code cells with saved outputs: 387
Image outputs: 88


In [5]:

# ============================================================
# SA 19C — DISSERTATION RESULTS PREVIEW
# Environmental Data Cube for Oglala Lakota
#
# ADD BELOW SA 19B
# Uses the existing search_records(), catalog,
# source_path and export_dir project location.
# Does not rerun Notebook 09 or modify source files.
# ============================================================

import base64
import html
import re
import zipfile
import csv
import io
from datetime import datetime
from pathlib import Path

import ipywidgets as widgets
from IPython.display import (
    display, HTML, Image, SVG,
    clear_output, FileLink
)

# ------------------------------------------------------------
# 1. PREVIEW CONTROLS
# ------------------------------------------------------------

preview_button = widgets.Button(
    description="Preview Matching Results",
    button_style="info",
    icon="eye",
    layout=widgets.Layout(width="245px")
)

previous_button = widgets.Button(
    description="Previous",
    icon="arrow-left",
    layout=widgets.Layout(width="125px")
)

next_button = widgets.Button(
    description="Next",
    icon="arrow-right",
    layout=widgets.Layout(width="125px")
)

figure_only = widgets.Checkbox(
    value=False,
    description="Show only cells containing figures",
    indent=False
)

preview_position = widgets.HTML()

preview_selector = widgets.Dropdown(
    options=[],
    description="Result:",
    style={"description_width": "65px"},
    layout=widgets.Layout(width="780px")
)

download_current = widgets.Button(
    description="Export This Result",
    button_style="danger",
    icon="download",
    layout=widgets.Layout(width="190px")
)

preview_output = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #E8D6D9",
        padding="16px",
        width="850px"
    )
)

download_output = widgets.Output()

preview_records = []


# ------------------------------------------------------------
# 2. FIND MATCHING RESULTS
# ------------------------------------------------------------

def load_preview(_=None):

    global preview_records

    preview_records = search_records()

    if figure_only.value:
        preview_records = [
            r for r in preview_records
            if r["images"] > 0
        ]

    options = []

    for index, record in enumerate(preview_records):

        label = (
            f'{record["section"]} | '
            f'Cell {record["cell"]:03d} | '
            f'{record["heading"][:48]} | '
            f'{record["images"]} figures'
        )

        options.append((label, index))

    preview_selector.options = options

    if options:
        preview_selector.value = 0
        show_preview()
    else:
        with preview_output:
            clear_output(wait=True)
            print(
                "No matching results. Change your search "
                "or uncheck the figure-only filter."
            )

        preview_position.value = "0 results"


# ------------------------------------------------------------
# 3. DISPLAY SELECTED RESULT
# ------------------------------------------------------------

def show_preview(change=None):

    if not preview_records:
        return

    position = preview_selector.value

    if position is None:
        return

    record = preview_records[position]

    preview_position.value = (
        f"<b>Result {position + 1} "
        f"of {len(preview_records)}</b>"
    )

    with preview_output:

        clear_output(wait=True)

        title = html.escape(record["heading"])

        display(HTML(f"""
        <div style="
            background:#742631;
            color:white;
            padding:18px;
            border-radius:12px;
            margin-bottom:16px;
        ">
            <div style="
                font-size:12px;
                color:#F0D2D5;
            ">
                {html.escape(record["section"])}
                · NOTEBOOK 09
            </div>

            <h3 style="
                color:white;
                margin:8px 0;
            ">
                {title}
            </h3>

            <div style="font-size:13px;">
                Original notebook cell:
                {record["cell"]}
            </div>
        </div>
        """))

        for output_number, output in enumerate(
            record["outputs"], start=1
        ):

            data = output.get("data", {})

            # Saved PNG figure
            if "image/png" in data:

                encoded = as_text(data["image/png"])

                display(Image(
                    data=base64.b64decode(encoded)
                ))

            # Saved JPEG figure
            elif "image/jpeg" in data:

                encoded = as_text(data["image/jpeg"])

                display(Image(
                    data=base64.b64decode(encoded),
                    format="jpeg"
                ))

            # Saved SVG figure
            elif "image/svg+xml" in data:

                display(SVG(
                    data=as_text(data["image/svg+xml"])
                ))

            # HTML tables
            elif "text/html" in data:

                display(HTML(
                    as_text(data["text/html"])
                ))

            # Printed statistical output
            elif output.get("output_type") == "stream":

                text = as_text(
                    output.get("text", "")
                )

                display(HTML(
                    '<pre style="'
                    'white-space:pre-wrap;'
                    'font-size:13px;'
                    'background:#FAF7F6;'
                    'padding:12px;'
                    'border-radius:8px;">'
                    + html.escape(text[:40000])
                    + '</pre>'
                ))

            # Plain text statistics
            elif "text/plain" in data:

                text = as_text(
                    data["text/plain"]
                )

                display(HTML(
                    '<pre style="'
                    'white-space:pre-wrap;'
                    'font-size:13px;'
                    'background:#FAF7F6;'
                    'padding:12px;'
                    'border-radius:8px;">'
                    + html.escape(text[:40000])
                    + '</pre>'
                ))


# ------------------------------------------------------------
# 4. PREVIOUS / NEXT NAVIGATION
# ------------------------------------------------------------

def go_previous(_):

    position = preview_selector.value

    if position is not None and position > 0:
        preview_selector.value = position - 1


def go_next(_):

    position = preview_selector.value

    if (
        position is not None
        and position < len(preview_records) - 1
    ):
        preview_selector.value = position + 1


previous_button.on_click(go_previous)
next_button.on_click(go_next)

preview_selector.observe(
    show_preview,
    names="value"
)

preview_button.on_click(load_preview)

figure_only.observe(
    load_preview,
    names="value"
)


# ------------------------------------------------------------
# 5. EXPORT ONLY THE CURRENT PREVIEWED RESULT
# ------------------------------------------------------------

def export_current_result(_):

    if not preview_records:
        return

    position = preview_selector.value

    if position is None:
        return

    record = preview_records[position]

    destination = (
        project / "results"
        / "SA19_dissertation_exports"
    )

    destination.mkdir(
        parents=True,
        exist_ok=True
    )

    timestamp = datetime.now().strftime(
        "%Y%m%d_%H%M%S_%f"
    )

    safe_section = re.sub(
        r"[^A-Za-z0-9._-]+",
        "_",
        record["section"]
    )

    zip_path = destination / (
        f"{safe_section}_cell_"
        f'{record["cell"]:03d}_{timestamp}.zip'
    )

    index_rows = []

    with zipfile.ZipFile(
        zip_path,
        "w",
        compression=zipfile.ZIP_DEFLATED
    ) as archive:

        for number, output in enumerate(
            record["outputs"], start=1
        ):

            data = output.get("data", {})

            filename = None
            contents = None

            if "image/png" in data:

                filename = f"figure_{number:02d}.png"

                contents = base64.b64decode(
                    as_text(data["image/png"])
                )

            elif "image/jpeg" in data:

                filename = f"figure_{number:02d}.jpg"

                contents = base64.b64decode(
                    as_text(data["image/jpeg"])
                )

            elif "image/svg+xml" in data:

                filename = f"figure_{number:02d}.svg"

                contents = as_text(
                    data["image/svg+xml"]
                )

            elif "text/html" in data:

                filename = f"table_{number:02d}.html"

                contents = as_text(
                    data["text/html"]
                )

            elif output.get("output_type") == "stream":

                filename = (
                    f"printed_statistics_{number:02d}.txt"
                )

                contents = as_text(
                    output.get("text", "")
                )

            elif "text/plain" in data:

                filename = (
                    f"statistics_{number:02d}.txt"
                )

                contents = as_text(
                    data["text/plain"]
                )

            if filename is not None:

                archive.writestr(
                    filename,
                    contents
                )

                index_rows.append({
                    "section": record["section"],
                    "cell": record["cell"],
                    "heading": record["heading"],
                    "filename": filename
                })

        # Save traceability information.
        buffer = io.StringIO()

        writer = csv.DictWriter(
            buffer,
            fieldnames=[
                "section", "cell",
                "heading", "filename"
            ]
        )

        writer.writeheader()
        writer.writerows(index_rows)

        archive.writestr(
            "source_index.csv",
            buffer.getvalue()
        )

        archive.writestr(
            "original_analysis_code.py",
            record["source"]
        )

    with download_output:

        clear_output(wait=True)

        print("Selected result exported.")
        print("Original cell:", record["cell"])
        print("Files:", len(index_rows))

        display(FileLink(
            str(zip_path),
            result_html_prefix=(
                "<b>Download this result:</b> "
            )
        ))


download_current.on_click(export_current_result)


# ------------------------------------------------------------
# 6. DISPLAY PREVIEW INTERFACE
# ------------------------------------------------------------

preview_header = widgets.HTML("""
<div style="
    background:linear-gradient(125deg,#681C27,#9B303D);
    color:white;
    padding:24px;
    border-radius:16px;
    margin-bottom:14px;
">
    <h2 style="color:white;margin:0 0 8px;">
        Dissertation Results Preview
    </h2>

    <p style="color:#F9E8E4;">
        Review figures, correlations, statistical tables,
        and validation outputs before downloading.
    </p>

    <small>
        SA 19C · Environmental Data Cube for Oglala Lakota
    </small>
</div>
""")

preview_interface = widgets.VBox([
    preview_header,
    widgets.HBox([
        preview_button,
        figure_only
    ]),
    preview_position,
    preview_selector,
    widgets.HBox([
        previous_button,
        next_button,
        download_current
    ]),
    preview_output,
    download_output
])

display(preview_interface)

# Load the current SA 19B search automatically.
load_preview()


In [6]:

# ============================================================
# SA 19D — COMPLETE DISSERTATION RESULTS LIBRARY
# Environmental Data Cube for Oglala Lakota
#
# Extends SA 19B and SA 19C.
# Includes ALL saved results from Notebook 09.
#
# READ-ONLY: Original notebook and data unchanged.
# ============================================================

import re
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ------------------------------------------------------------
# 1. REBUILD SECTION INDEX FOR ALL NOTEBOOK RESULTS
# ------------------------------------------------------------

def rebuild_complete_catalog():

    global catalog

    section_re = re.compile(
        r"\b(?:SA|QA)\s*[-:]?\s*"
        r"(\d+(?:\.\d+)?[A-Za-z]?)\b",
        re.IGNORECASE
    )

    complete = []

    current_section = "General"
    current_heading = "Unlabeled analysis"

    for cell_number, cell in enumerate(
        source_cells, start=1
    ):

        source = as_text(
            cell.get("source", "")
        )

        if cell.get("cell_type") == "markdown":

            heading_lines = [
                line.strip().lstrip("#").strip()
                for line in source.splitlines()
                if line.strip().startswith("#")
            ]

            if heading_lines:

                current_heading = heading_lines[0]

                match = section_re.search(
                    current_heading
                )

                if match:
                    prefix = re.search(
                        r"\b(SA|QA)\b",
                        current_heading,
                        re.IGNORECASE
                    ).group(1).upper()

                    current_section = (
                        prefix + " "
                        + match.group(1).upper()
                    )

            continue

        if cell.get("cell_type") != "code":
            continue

        outputs = cell.get("outputs", [])

        if not outputs:
            continue

        image_count = 0
        table_count = 0
        text_count = 0
        output_text = []

        for output in outputs:

            data = output.get("data", {})

            if any(
                key in data
                for key in (
                    "image/png",
                    "image/jpeg",
                    "image/svg+xml"
                )
            ):
                image_count += 1

            if "text/html" in data:
                table_count += 1

            if (
                "text/plain" in data
                or output.get("output_type") == "stream"
            ):
                text_count += 1

            output_text.extend([
                as_text(
                    output.get("text", "")
                ),
                as_text(
                    data.get("text/plain", "")
                )
            ])

        if not (
            image_count or table_count or text_count
        ):
            continue

        searchable = "\n".join([
            current_section,
            current_heading,
            source,
            *output_text
        ]).lower()

        complete.append({
            "cell": cell_number,
            "section": current_section,
            "heading": current_heading,
            "source": source,
            "outputs": outputs,
            "search": searchable,
            "images": image_count,
            "html_tables": table_count,
            "text_count": text_count
        })

    catalog = complete

    return complete


complete_catalog = rebuild_complete_catalog()

print("Complete results catalog loaded.")
print("Cells with results:", len(complete_catalog))
print(
    "Saved figures:",
    sum(r["images"] for r in complete_catalog)
)
print(
    "HTML table/output records:",
    sum(r["html_tables"] for r in complete_catalog)
)


# ------------------------------------------------------------
# 2. CREATE FULL-LIBRARY SEARCH CONTROLS
# ------------------------------------------------------------

all_sections = sorted({
    r["section"]
    for r in complete_catalog
})

library_scope = widgets.Dropdown(
    options=[
        ("Entire Notebook 09", "all"),
        ("SA analyses only", "SA"),
        ("QA analyses only", "QA"),
        ("General/unassigned results", "General")
    ],
    value="all",
    description="Scope:",
    style={"description_width": "85px"},
    layout=widgets.Layout(width="500px")
)

library_category = widgets.Dropdown(
    options=[
        ("Everything", "all"),
        ("Figures and maps", "figures"),
        ("Box plots", "boxplots"),
        ("Correlations and regression", "correlation"),
        ("Statistical tables", "tables"),
        ("Validation and reliability", "validation"),
        ("Spatial statistics", "spatial"),
        ("Climate and vegetation", "climate"),
        ("Sensitivity and uncertainty", "sensitivity")
    ],
    value="all",
    description="Category:",
    style={"description_width": "85px"},
    layout=widgets.Layout(width="500px")
)

library_search = widgets.Text(
    value="",
    placeholder=(
        "Search by section, statistic, variable, "
        "plot type, or code keyword..."
    ),
    description="Search:",
    style={"description_width": "85px"},
    layout=widgets.Layout(width="750px")
)

library_find_button = widgets.Button(
    description="Find and Preview",
    button_style="info",
    icon="eye",
    layout=widgets.Layout(width="190px")
)

library_status = widgets.Output()


# ------------------------------------------------------------
# 3. CATEGORY SEARCH TERMS
# ------------------------------------------------------------

category_terms = {

    "boxplots": [
        "boxplot",
        "box plot",
        "boxenplot",
        "box-and-whisker",
        "quartile"
    ],

    "correlation": [
        "correlation",
        "pearson",
        "spearman",
        "corrcoef",
        "regression",
        "linregress",
        "ols",
        "r-squared"
    ],

    "validation": [
        "validation",
        "reliability",
        "accuracy",
        "quality",
        "agreement",
        "completeness"
    ],

    "spatial": [
        "moran",
        "geary",
        "spatial autocorrelation",
        "spatial correlation",
        "block bootstrap"
    ],

    "climate": [
        "precip",
        "temperature",
        "tmax",
        "ndvi",
        "ndmi",
        "evi",
        "climate"
    ],

    "sensitivity": [
        "sensitivity",
        "uncertainty",
        "bootstrap",
        "confidence interval",
        "threshold",
        "seasonal window"
    ]
}


# ------------------------------------------------------------
# 4. CONNECT COMPLETE LIBRARY TO EXISTING PREVIEW
# ------------------------------------------------------------

def find_complete_library(_=None):

    global preview_records

    scope = library_scope.value
    category = library_category.value
    query = library_search.value.strip().lower()

    matches = []

    for record in complete_catalog:

        section = record["section"]
        searchable = record["search"]

        # Notebook section scope
        if scope == "SA" and not section.startswith("SA"):
            continue

        if scope == "QA" and not section.startswith("QA"):
            continue

        if (
            scope == "General"
            and section != "General"
        ):
            continue

        # Category filtering
        if category == "figures":

            if record["images"] == 0:
                continue

        elif category == "tables":

            if record["html_tables"] == 0:
                continue

        elif category != "all":

            terms = category_terms.get(
                category, []
            )

            if not any(
                term in searchable
                for term in terms
            ):
                continue

        # Free text search
        if query and query not in searchable:
            continue

        matches.append(record)

    preview_records = matches

    options = [
        (
            f'{r["section"]} | '
            f'Cell {r["cell"]:03d} | '
            f'{r["heading"][:48]} | '
            f'{r["images"]} figures',
            i
        )
        for i, r in enumerate(matches)
    ]

    preview_selector.options = options

    with library_status:

        clear_output(wait=True)

        print("COMPLETE DISSERTATION LIBRARY")
        print("=" * 42)
        print("Matching result cells:", len(matches))
        print(
            "Matching saved figures:",
            sum(r["images"] for r in matches)
        )
        print(
            "Matching HTML outputs:",
            sum(r["html_tables"] for r in matches)
        )

    if options:

        preview_selector.value = 0
        show_preview()

    else:

        preview_position.value = "No matching results"

        with preview_output:
            clear_output(wait=True)
            print(
                "No results match these filters. "
                "Try Everything or clear the search."
            )


library_find_button.on_click(
    find_complete_library
)


# ------------------------------------------------------------
# 5. DISPLAY COMPLETE LIBRARY
# ------------------------------------------------------------

library_header = widgets.HTML("""
<div style="
    background:linear-gradient(125deg,#681C27,#9B303D);
    color:white;
    border-radius:16px;
    padding:25px;
    margin-bottom:14px;
">
    <div style="
        font-size:12px;
        letter-spacing:1px;
        color:#F2D5D7;
    ">
        ENVIRONMENTAL DATA CUBE FOR OGLALA LAKOTA
    </div>

    <h2 style="
        color:white;
        margin:10px 0;
    ">
        Complete Dissertation Results Library
    </h2>

    <p style="color:#F9E8E4;">
        SA 19D — All figures, statistics,
        correlations, and validation results
    </p>

    <small>
        Search → Preview → Export
    </small>
</div>
""")

complete_library_interface = widgets.VBox([
    library_header,
    library_scope,
    library_category,
    library_search,
    library_find_button,
    library_status,
    preview_position,
    preview_selector,
    widgets.HBox([
        previous_button,
        next_button,
        download_current
    ]),
    preview_output,
    download_output
])

display(complete_library_interface)

find_complete_library()


Complete results catalog loaded.
Cells with results: 384
Saved figures: 88
HTML table/output records: 566


In [7]:

# ============================================================
# SA 19E — MAPS & SPATIAL ANALYSIS GALLERY
# Environmental Data Cube for Oglala Lakota
#
# Add below SA 19D.
# Reads existing saved figures from Notebook 09.
# Does not rerun analyses or modify research files.
# ============================================================

import re
import io
import csv
import json
import base64
import zipfile
from pathlib import Path
from datetime import datetime

import ipywidgets as widgets
from IPython.display import display, HTML, Image, SVG, FileLink, clear_output


# ------------------------------------------------------------
# 1. MAP LABELS FROM ORIGINAL NOTEBOOK
# ------------------------------------------------------------

known_maps = {
    294: "Wanzi Parcel Boundary Validation",
    296: "NDMI Change Map — 2019–2022",
    297: "NDMI Change Classification",
    299: "NDMI Recovery Assessment",
    301: "Vegetation Trajectory Map",
    309: "Revised Vegetation Assessment",
    344: "Spatial Heterogeneity of Vegetation Recovery",
    348: "Spatially Balanced Field-Sampling Candidates"
}

map_terms = [
    "imshow", "pcolormesh", "geopandas",
    ".plot(", "plot_map", "raster",
    "spatial map", "change map",
    "trajectory map", "classification map",
    "parcel map", "sampling map",
    "boundary map", "spatial distribution"
]


# ------------------------------------------------------------
# 2. COLLECT SAVED MAPS
# ------------------------------------------------------------

def get_image_outputs(record):

    images = []

    for output_index, output in enumerate(
        record["outputs"], start=1
    ):
        data = output.get("data", {})

        for mime, ext in [
            ("image/png", "png"),
            ("image/jpeg", "jpg"),
            ("image/svg+xml", "svg")
        ]:
            if mime in data:
                images.append({
                    "output_index": output_index,
                    "mime": mime,
                    "extension": ext,
                    "content": as_text(data[mime])
                })
                break

    return images


map_catalog = []

for record in complete_catalog:

    if record["images"] == 0:
        continue

    source = record["source"].lower()
    heading = record["heading"].lower()
    cell_number = record["cell"]

    is_map = (
        cell_number in known_maps
        or any(
            term in source or term in heading
            for term in map_terms
        )
    )

    if not is_map:
        continue

    images = get_image_outputs(record)

    for image_number, image_data in enumerate(
        images, start=1
    ):

        label = known_maps.get(
            cell_number,
            record["heading"]
        )

        if len(images) > 1:
            label += f" — Figure {image_number}"

        map_catalog.append({
            "cell": cell_number,
            "section": record["section"],
            "label": label,
            "heading": record["heading"],
            "source": record["source"],
            "image": image_data
        })

print("Maps/spatial figures found:", len(map_catalog))


# ------------------------------------------------------------
# 3. GALLERY CONTROLS
# ------------------------------------------------------------

map_search = widgets.Text(
    placeholder="Search NDVI, NDMI, recovery, sampling...",
    description="Search:",
    style={"description_width": "70px"},
    layout=widgets.Layout(width="650px")
)

map_select = widgets.SelectMultiple(
    options=[],
    rows=12,
    description="Maps:",
    style={"description_width": "70px"},
    layout=widgets.Layout(width="800px", height="290px")
)

map_status = widgets.HTML()

preview_maps_button = widgets.Button(
    description="Preview Selected Maps",
    button_style="info",
    icon="eye",
    layout=widgets.Layout(width="220px")
)

select_all_maps_button = widgets.Button(
    description="Select All Matches",
    icon="check-square-o",
    layout=widgets.Layout(width="190px")
)

export_maps_button = widgets.Button(
    description="Export Selected Maps",
    button_style="danger",
    icon="download",
    layout=widgets.Layout(width="210px")
)

map_preview_output = widgets.Output()
map_export_output = widgets.Output()

visible_maps = []


# ------------------------------------------------------------
# 4. FILTER MAP COLLECTION
# ------------------------------------------------------------

def update_map_list(change=None):

    global visible_maps

    query = map_search.value.strip().lower()

    visible_maps = [
        record
        for record in map_catalog
        if not query
        or query in (
            record["label"] + " "
            + record["heading"] + " "
            + record["section"] + " "
            + record["source"]
        ).lower()
    ]

    map_select.options = [
        (
            f'{record["section"]} | '
            f'Cell {record["cell"]:03d} | '
            f'{record["label"][:65]}',
            index
        )
        for index, record in enumerate(visible_maps)
    ]

    map_status.value = (
        f"<b>{len(visible_maps)} matching maps "
        f"and spatial figures</b>"
    )


def select_all_maps(_):

    map_select.value = tuple(
        range(len(visible_maps))
    )


map_search.observe(
    update_map_list,
    names="value"
)

select_all_maps_button.on_click(
    select_all_maps
)


# ------------------------------------------------------------
# 5. PREVIEW MULTIPLE MAPS AT ONCE
# ------------------------------------------------------------

def preview_selected_maps(_):

    selected = list(map_select.value)

    with map_preview_output:

        clear_output(wait=True)

        if not selected:
            print("Select one or more maps first.")
            return

        print(
            f"Previewing {len(selected)} "
            "selected maps\n"
        )

        for index in selected:

            record = visible_maps[index]
            image_data = record["image"]

            display(HTML(
                '<div style="background:#742631;'
                'color:white;padding:12px;'
                'border-radius:8px;'
                'margin:16px 0 10px;">'
                f'<b>{record["label"]}</b>'
                f'<div style="font-size:12px;">'
                f'{record["section"]} | '
                f'Original Cell {record["cell"]}'
                '</div></div>'
            ))

            if image_data["extension"] == "svg":

                display(SVG(
                    data=image_data["content"]
                ))

            else:

                display(Image(
                    data=base64.b64decode(
                        image_data["content"]
                    )
                ))


preview_maps_button.on_click(
    preview_selected_maps
)


# ------------------------------------------------------------
# 6. EXPORT SELECTED MAPS AS A ZIP
# ------------------------------------------------------------

def export_selected_maps(_):

    selected = list(map_select.value)

    with map_export_output:

        clear_output(wait=True)

        if not selected:
            print("Select one or more maps first.")
            return

        export_folder = (
            project / "results"
            / "SA19_dissertation_exports"
            / "maps"
        )

        export_folder.mkdir(
            parents=True,
            exist_ok=True
        )

        timestamp = datetime.now().strftime(
            "%Y%m%d_%H%M%S"
        )

        zip_path = (
            export_folder
            / f"wanzi_selected_maps_{timestamp}.zip"
        )

        index_rows = []

        with zipfile.ZipFile(
            zip_path,
            "w",
            zipfile.ZIP_DEFLATED
        ) as archive:

            for index in selected:

                record = visible_maps[index]
                image_data = record["image"]

                safe_label = re.sub(
                    r"[^A-Za-z0-9_-]+",
                    "_",
                    record["label"]
                ).strip("_")[:65]

                filename = (
                    f'cell_{record["cell"]:03d}_'
                    f'{image_data["output_index"]:02d}_'
                    f'{safe_label}.'
                    f'{image_data["extension"]}'
                )

                if image_data["extension"] == "svg":

                    content = image_data["content"]

                else:

                    content = base64.b64decode(
                        image_data["content"]
                    )

                archive.writestr(
                    filename,
                    content
                )

                index_rows.append({
                    "filename": filename,
                    "label": record["label"],
                    "section": record["section"],
                    "notebook_cell": record["cell"],
                    "source_notebook": source_path.name
                })

            buffer = io.StringIO()

            writer = csv.DictWriter(
                buffer,
                fieldnames=[
                    "filename",
                    "label",
                    "section",
                    "notebook_cell",
                    "source_notebook"
                ]
            )

            writer.writeheader()
            writer.writerows(index_rows)

            archive.writestr(
                "map_figure_index.csv",
                buffer.getvalue()
            )

        print(
            f"Exported {len(selected)} maps."
        )

        print("Saved to:", zip_path)

        display(FileLink(str(zip_path)))


export_maps_button.on_click(
    export_selected_maps
)


# ------------------------------------------------------------
# 7. DISPLAY MAP GALLERY
# ------------------------------------------------------------

map_header = widgets.HTML("""
<div style="
    background:linear-gradient(125deg,#681C27,#9B303D);
    color:white;
    border-radius:16px;
    padding:24px;
    margin-bottom:14px;
">
    <h2 style="color:white;margin:0 0 8px;">
        Wanzi Maps & Spatial Analysis Gallery
    </h2>

    <p style="color:#F9E8E4;">
        Preview multiple dissertation maps together
        and export only the figures you select.
    </p>

    <small>
        SA 19E · Environmental Data Cube for Oglala Lakota
    </small>
</div>
""")

map_gallery = widgets.VBox([
    map_header,
    map_search,
    map_status,
    map_select,
    widgets.HBox([
        select_all_maps_button,
        preview_maps_button,
        export_maps_button
    ]),
    map_preview_output,
    map_export_output
])

update_map_list()
display(map_gallery)


Maps/spatial figures found: 75


In [8]:
# ======================================================================
# ENVIRONMENTAL DATA CUBE FOR OGLALA LAKOTA
# VIEW CONTENTS OF NOTEBOOK 11 — DISSERTATION FIGURES GALLERY
#
# READ-ONLY: Does not change Notebook 11 or any source files.
# ======================================================================

from pathlib import Path
import re
import shlex
import nbformat
import pandas as pd
from IPython.display import display, Markdown, Image

# ----------------------------------------------------------------------
# 1. LOCATE EXISTING GALLERY NOTEBOOK
# ----------------------------------------------------------------------

project = (
    Path.home()
    / "Documents"
    / "ElishaWakinyanZi-WORK"
    / "datacube_2026"
    / "ate_makoce_agcube"
)

gallery_path = (
    project
    / "Notebooks"
    / "11_dissertation_figures_stats_results_gallery.ipynb"
)

if not gallery_path.exists():
    raise FileNotFoundError(
        f"Gallery notebook not found:\n{gallery_path}"
    )

gallery = nbformat.read(gallery_path, as_version=4)

display(Markdown(
    "# Environmental Data Cube for Oglala Lakota\n"
    "## Dissertation Figures, Statistics & Results Gallery\n"
    f"**Notebook:** `{gallery_path.name}`\n\n"
    f"**Total cells:** {len(gallery.cells)}"
))

# ----------------------------------------------------------------------
# 2. INVENTORY ALL NOTEBOOK CELLS
# ----------------------------------------------------------------------

inventory = []

for i, cell in enumerate(gallery.cells, start=1):

    source = cell.get("source", "")

    headings = re.findall(
        r"(?m)^#{1,6}\s+(.+)$",
        source
    )

    title = (
        headings[0]
        if headings
        else source.strip().splitlines()[0][:100]
        if source.strip()
        else "(empty)"
    )

    inventory.append({
        "Cell": i,
        "Type": cell.cell_type,
        "Title / Preview": title,
        "Saved Outputs": len(cell.get("outputs", []))
            if cell.cell_type == "code" else 0
    })

display(Markdown("### Notebook cell inventory"))
display(pd.DataFrame(inventory))

# ----------------------------------------------------------------------
# 3. DISPLAY FIGURES REFERENCED IN MARKDOWN
# ----------------------------------------------------------------------

display(Markdown("---\n## Gallery Figures"))

image_pattern = re.compile(
    r"!\[([^\]]*)\]\(([^)]+)\)"
)

shown_images = set()
image_count = 0
missing_images = []

for cell in gallery.cells:

    if cell.cell_type != "markdown":
        continue

    for alt, link in image_pattern.findall(cell.source):

        # Remove optional Markdown title.
        raw_link = link.strip()

        if raw_link.startswith("<") and ">" in raw_link:
            raw_link = raw_link[1:raw_link.index(">")]
        else:
            try:
                raw_link = shlex.split(raw_link)[0]
            except (ValueError, IndexError):
                raw_link = raw_link.split(" ")[0]

        # Skip remote images; this viewer uses local files.
        if raw_link.startswith(("http://", "https://", "data:")):
            print(f"External image link: {raw_link}")
            continue

        image_path = (
            gallery_path.parent / raw_link
        ).resolve()

        if image_path in shown_images:
            continue

        shown_images.add(image_path)

        if not image_path.exists():
            missing_images.append(str(image_path))
            continue

        if image_path.suffix.lower() in {
            ".png", ".jpg", ".jpeg", ".gif"
        }:
            display(Markdown(f"### {alt or image_path.stem}"))
            display(Image(filename=str(image_path), width=950))
            image_count += 1
        else:
            print(f"Linked image asset: {image_path.name}")

# ----------------------------------------------------------------------
# 4. DISPLAY STATISTICAL TABLES FROM LINKED CSV FILES
# ----------------------------------------------------------------------

display(Markdown("---\n## Gallery Statistical Tables"))

link_pattern = re.compile(
    r"(?<!!)\[([^\]]+)\]\(([^)]+\.csv)\)"
)

shown_csvs = set()
table_count = 0

for cell in gallery.cells:

    if cell.cell_type != "markdown":
        continue

    for label, link in link_pattern.findall(cell.source):

        csv_path = (
            gallery_path.parent / link
        ).resolve()

        if csv_path in shown_csvs or not csv_path.exists():
            continue

        shown_csvs.add(csv_path)

        display(Markdown(f"### {label}"))

        try:
            df = pd.read_csv(csv_path)

            print(
                f"{csv_path.name} | "
                f"{len(df):,} rows × {len(df.columns)} columns"
            )

            display(df.head(20))
            table_count += 1

        except Exception as exc:
            print(f"Could not read {csv_path.name}: {exc}")

# ----------------------------------------------------------------------
# 5. DISPLAY ADDITIONAL DATASET LINKS
# ----------------------------------------------------------------------

display(Markdown("---\n## Linked Spatial Datasets"))

spatial_pattern = re.compile(
    r"\[([^\]]+)\]\(([^)]+\.(?:tif|tiff|nc|gpkg|shp))\)",
    flags=re.IGNORECASE
)

spatial_count = 0
shown_spatial = set()

for cell in gallery.cells:

    if cell.cell_type != "markdown":
        continue

    for label, link in spatial_pattern.findall(cell.source):

        data_path = (
            gallery_path.parent / link
        ).resolve()

        if data_path in shown_spatial:
            continue

        shown_spatial.add(data_path)

        status = "AVAILABLE" if data_path.exists() else "MISSING"

        print(f"{status} | {label}")
        print(f"  {data_path}")

        spatial_count += 1

# ----------------------------------------------------------------------
# 6. DISPLAY SAVED OUTPUTS ALREADY EMBEDDED IN NOTEBOOK
# ----------------------------------------------------------------------

display(Markdown("---\n## Saved Notebook Outputs"))

saved_output_cells = [
    (i, cell)
    for i, cell in enumerate(gallery.cells, start=1)
    if cell.cell_type == "code" and cell.get("outputs")
]

if saved_output_cells:
    for cell_number, cell in saved_output_cells:
        print(
            f"Cell {cell_number}: "
            f"{len(cell.outputs)} saved output(s)"
        )
else:
    print("No saved code-cell outputs found.")

# ----------------------------------------------------------------------
# 7. SUMMARY
# ----------------------------------------------------------------------

display(Markdown("---\n## Gallery Summary"))

summary = pd.DataFrame([
    ["Notebook cells", len(gallery.cells)],
    ["Local figures displayed", image_count],
    ["CSV tables displayed", table_count],
    ["Spatial datasets linked", spatial_count],
    ["Cells with saved outputs", len(saved_output_cells)],
    ["Missing image files", len(missing_images)]
], columns=["Gallery Content", "Count"])

display(summary)

if missing_images:
    display(Markdown("### Missing image references"))
    for path in missing_images:
        print(path)

print("\nGallery inspection complete.")
print("Notebook 11 and all research files remain unchanged.")

# Environmental Data Cube for Oglala Lakota
## Dissertation Figures, Statistics & Results Gallery
**Notebook:** `11_dissertation_figures_stats_results_gallery.ipynb`

**Total cells:** 6

### Notebook cell inventory

,Cell,Type,Title / Preview,Saved Outputs
0,1,code,"message = ""Han, Mitakuyap!""",1
1,2,code,==============================================...,2
2,3,code,==============================================...,2
3,4,code,==============================================...,1
4,5,code,==============================================...,2
5,6,code,==============================================...,2


---
## Gallery Figures

---
## Gallery Statistical Tables

...

---
## Saved Notebook Outputs

Cell 1: 1 saved output(s)
Cell 2: 2 saved output(s)
Cell 3: 2 saved output(s)
Cell 4: 1 saved output(s)
Cell 5: 2 saved output(s)
Cell 6: 2 saved output(s)


---
## Gallery Summary

,Gallery Content,Count
0,Notebook cells,6
1,Local figures displayed,0
2,CSV tables displayed,0
3,Spatial datasets linked,0
4,Cells with saved outputs,6
5,Missing image files,0



Gallery inspection complete.
Notebook 11 and all research files remain unchanged.


In [9]:
from pathlib import Path
from datetime import datetime
import nbformat

project = (
    Path.home()
    / "Documents"
    / "ElishaWakinyanZi-WORK"
    / "datacube_2026"
    / "ate_makoce_agcube"
)

notebooks = project / "Notebooks"

target = (
    notebooks
    / "11_dissertation_figures_stats_results_gallery.ipynb"
)

candidates = [target] + sorted(
    notebooks.glob(
        "11_dissertation_figures_stats_results_gallery_backup*.ipynb"
    )
)

print("=" * 75)
print("NOTEBOOK 11 — GALLERY RECOVERY DIAGNOSTIC")
print("=" * 75)

for path in candidates:

    if not path.exists():
        continue

    try:
        nb = nbformat.read(path, as_version=4)

        sources = "\n".join(
            cell.source for cell in nb.cells
        )

        has_gallery = (
            "BEGIN ATE_MACOKE_MANAGED_GALLERY" in sources
        )

        modified = datetime.fromtimestamp(
            path.stat().st_mtime
        )

        print(f"\nFILE: {path.name}")
        print(f"Modified: {modified}")
        print(f"Cells: {len(nb.cells)}")
        print(f"Ate_Macoke gallery: {'YES' if has_gallery else 'NO'}")
        print(f"Size: {path.stat().st_size:,} bytes")

    except Exception as exc:
        print(f"\nCould not read {path.name}: {exc}")

print("\nSOURCE FIGURES")

source = project / "results" / "SA20_satellite_maps"

for folder_name in [
    "Ate_Macoke_Aerial_Imagery",
    "Ate_Macoke_Community_Land_Maps",
    "Ate_Macoke_NDVI_Trend_Analysis",
]:

    folder = source / folder_name

    png_count = (
        len(list(folder.rglob("*.png")))
        if folder.exists()
        else 0
    )

    print(f"{folder_name}: {png_count} PNG files")

print("\nNo files were changed.")


NOTEBOOK 11 — GALLERY RECOVERY DIAGNOSTIC

FILE: 11_dissertation_figures_stats_results_gallery.ipynb
Modified: 2026-10-16 21:10:50.831951
Cells: 6
Ate_Macoke gallery: NO
Size: 91,487 bytes

FILE: 11_dissertation_figures_stats_results_gallery_backup_before_Ate_Macoke_20261016_210918_743149.ipynb
Modified: 2026-10-16 19:27:28.275961
Cells: 6
Ate_Macoke gallery: NO
Size: 91,487 bytes

SOURCE FIGURES
Ate_Macoke_Aerial_Imagery: 1 PNG files
Ate_Macoke_Community_Land_Maps: 2 PNG files
Ate_Macoke_NDVI_Trend_Analysis: 1 PNG files

No files were changed.


<!-- BEGIN ATE_MACOKE_MANAGED_GALLERY -->

# Ate_Macoke — Dissertation Figures and Statistical Results

**Environmental Data Cube for Oglala Lakota**

**Study period:** 2015–2023  
**Imagery:** Landsat growing-season NDVI, USDA/USGS aerial imagery  
**Study area:** Ate_Macoke, Pine Ridge Indian Reservation

This gallery presents vegetation-change figures, aerial imagery,
spatial trend analysis, and supporting statistical results.

---


## Aerial Imagery

### Ate Macoke Aerial Map 20261016 205950 341947

![Ate Macoke Aerial Map 20261016 205950 341947](<../results/SA20_satellite_maps/Ate_Macoke_Aerial_Imagery/Ate_Macoke_Aerial_Map_20261016_205950_341947.png>)

**Original figure:** [Ate_Macoke_Aerial_Map_20261016_205950_341947.png](<../results/SA20_satellite_maps/Ate_Macoke_Aerial_Imagery/Ate_Macoke_Aerial_Map_20261016_205950_341947.png>)

## Community Maps

### Ate Macoke Community Map 20261016 205527 337603

![Ate Macoke Community Map 20261016 205527 337603](<../results/SA20_satellite_maps/Ate_Macoke_Community_Land_Maps/Ate_Macoke_Community_Map_20261016_205527_337603.png>)

**Original figure:** [Ate_Macoke_Community_Map_20261016_205527_337603.png](<../results/SA20_satellite_maps/Ate_Macoke_Community_Land_Maps/Ate_Macoke_Community_Map_20261016_205527_337603.png>)

### Ate Macoke Four Panel 20261016 210103 365962

![Ate Macoke Four Panel 20261016 210103 365962](<../results/SA20_satellite_maps/Ate_Macoke_Community_Land_Maps/Ate_Macoke_Four_Panel_20261016_210103_365962.png>)

**Original figure:** [Ate_Macoke_Four_Panel_20261016_210103_365962.png](<../results/SA20_satellite_maps/Ate_Macoke_Community_Land_Maps/Ate_Macoke_Four_Panel_20261016_210103_365962.png>)

## NDVI Trend Analysis

### Ate Macoke NDVI Trend 20261016 210357 745832

![Ate Macoke NDVI Trend 20261016 210357 745832](<../results/SA20_satellite_maps/Ate_Macoke_NDVI_Trend_Analysis/Ate_Macoke_NDVI_Trend_20261016_210357_745832.png>)

**Original figure:** [Ate_Macoke_NDVI_Trend_20261016_210357_745832.png](<../results/SA20_satellite_maps/Ate_Macoke_NDVI_Trend_Analysis/Ate_Macoke_NDVI_Trend_20261016_210357_745832.png>)

## Additional Satellite Maps

### Ate Macoke Accurate Satellite Map

![Ate Macoke Accurate Satellite Map](<../results/SA20_satellite_maps/Ate_Macoke_Accurate_Satellite_Map.png>)

### Ate Macoke Accurate Satellite Map v2

![Ate Macoke Accurate Satellite Map v2](<../results/SA20_satellite_maps/Ate_Macoke_Accurate_Satellite_Map_v2.png>)

### Ate Macoke Accurate Satellite Map v3

![Ate Macoke Accurate Satellite Map v3](<../results/SA20_satellite_maps/Ate_Macoke_Accurate_Satellite_Map_v3.png>)

### Ate Macoke High Visibility Vegetation Map

![Ate Macoke High Visibility Vegetation Map](<../results/SA20_satellite_maps/Ate_Macoke_High_Visibility_Vegetation_Map.png>)

### Ate Macoke Landsat NDVI 2015 2023 Atlas

![Ate Macoke Landsat NDVI 2015 2023 Atlas](<../results/SA20_satellite_maps/Ate_Macoke_Landsat_NDVI_2015_2023_Atlas.png>)

### Ate Macoke NDVI Spatial 2019 2022 2023

![Ate Macoke NDVI Spatial 2019 2022 2023](<../results/SA20_satellite_maps/Ate_Macoke_NDVI_Spatial_2019_2022_2023.png>)

### Ate Macoke NDVI Through Time 2019 2023

![Ate Macoke NDVI Through Time 2019 2023](<../results/SA20_satellite_maps/Ate_Macoke_NDVI_Through_Time_2019_2023.png>)

### Ate Macoke NDVI Through Time Atlas

![Ate Macoke NDVI Through Time Atlas](<../results/SA20_satellite_maps/Ate_Macoke_NDVI_Through_Time_Atlas.png>)

### Ate Macoke Nine Year NDVI Atlas

![Ate Macoke Nine Year NDVI Atlas](<../results/SA20_satellite_maps/Ate_Macoke_Nine_Year_NDVI_Atlas.png>)

### Ate Macoke Original 15 Point Sampling Lattice

![Ate Macoke Original 15 Point Sampling Lattice](<../results/SA20_satellite_maps/Ate_Macoke_Original_15_Point_Sampling_Lattice.png>)

### Ate Macoke Original 15 Point Sampling Lattice RGB

![Ate Macoke Original 15 Point Sampling Lattice RGB](<../results/SA20_satellite_maps/Ate_Macoke_Original_15_Point_Sampling_Lattice_RGB.png>)

### Ate Macoke Professional Satellite Vegetation Map

![Ate Macoke Professional Satellite Vegetation Map](<../results/SA20_satellite_maps/Ate_Macoke_Professional_Satellite_Vegetation_Map.png>)

### Ate Macoke Refined Satellite Map

![Ate Macoke Refined Satellite Map](<../results/SA20_satellite_maps/Ate_Macoke_Refined_Satellite_Map.png>)

### Ate Macoke Satellite Three Class Map

![Ate Macoke Satellite Three Class Map](<../results/SA20_satellite_maps/Ate_Macoke_Satellite_Three_Class_Map.png>)

### Ate Macoke Satellite Vegetation Map 58pct Opacity

![Ate Macoke Satellite Vegetation Map 58pct Opacity](<../results/SA20_satellite_maps/Ate_Macoke_Satellite_Vegetation_Map_58pct_Opacity.png>)

### Ate Macoke Three Class Vegetation Map

![Ate Macoke Three Class Vegetation Map](<../results/SA20_satellite_maps/Ate_Macoke_Three_Class_Vegetation_Map.png>)

### Ate Macoke True Color Composite Comparison

![Ate Macoke True Color Composite Comparison](<../results/SA20_satellite_maps/Ate_Macoke_True_Color_Composite_Comparison.png>)

### Ate Macoke Verified Vegetation Trajectories

![Ate Macoke Verified Vegetation Trajectories](<../results/SA20_satellite_maps/Ate_Macoke_Verified_Vegetation_Trajectories.png>)


## Nine-Year NDVI Trend — 2015–2023

### Parcel-level trend

| Statistic | Result |
|---|---:|
| OLS slope | −0.00684 NDVI/year |
| OLS R² | 0.1130 |
| OLS p-value | 0.3765 |
| Theil–Sen slope | −0.00739 NDVI/year |
| Theil–Sen 95% CI | [−0.02603, +0.01021] |

### Spatial trends

| Category | Area (ha) | Percent |
|---|---:|---:|
| Declining | 143.73 | 55.5% |
| Relatively stable | 107.37 | 41.5% |
| Increasing | 7.92 | 3.1% |
| **Total analyzed** | **259.02** | **100%** |

The parcel-wide trend is not statistically significant
(p = 0.3765). Spatial classes use descriptive Theil–Sen
slope thresholds of ±0.005 NDVI/year, not statistical
significance tests.

Declining NDVI alone does not establish land degradation
or overgrazing.


### Annual NDVI Statistics

**Data:** [Ate_Macoke_Annual_NDVI_20261016_210357_745832.csv](<../results/SA20_satellite_maps/Ate_Macoke_NDVI_Trend_Analysis/Ate_Macoke_Annual_NDVI_20261016_210357_745832.csv>)

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

csv_path = Path('/Users/elishawakinyanzi/Documents/ElishaWakinyanZi-WORK/datacube_2026/ate_makoce_agcube/results/SA20_satellite_maps/Ate_Macoke_NDVI_Trend_Analysis/Ate_Macoke_Annual_NDVI_20261016_210357_745832.csv')
df = pd.read_csv(csv_path)
print(f'{len(df):,} rows × {len(df.columns)} columns')
display(df.head(20))

### Land-Cover Trend Statistics

**Data:** [Ate_Macoke_LandCover_Trend_20261016_210357_745832.csv](<../results/SA20_satellite_maps/Ate_Macoke_NDVI_Trend_Analysis/Ate_Macoke_LandCover_Trend_20261016_210357_745832.csv>)

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

csv_path = Path('/Users/elishawakinyanzi/Documents/ElishaWakinyanZi-WORK/datacube_2026/ate_makoce_agcube/results/SA20_satellite_maps/Ate_Macoke_NDVI_Trend_Analysis/Ate_Macoke_LandCover_Trend_20261016_210357_745832.csv')
df = pd.read_csv(csv_path)
print(f'{len(df):,} rows × {len(df.columns)} columns')
display(df.head(20))

### Pixel-Level Trend Statistics

**Data:** [Ate_Macoke_Pixel_Trend_20261016_210357_745832.csv](<../results/SA20_satellite_maps/Ate_Macoke_NDVI_Trend_Analysis/Ate_Macoke_Pixel_Trend_20261016_210357_745832.csv>)

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

csv_path = Path('/Users/elishawakinyanzi/Documents/ElishaWakinyanZi-WORK/datacube_2026/ate_makoce_agcube/results/SA20_satellite_maps/Ate_Macoke_NDVI_Trend_Analysis/Ate_Macoke_Pixel_Trend_20261016_210357_745832.csv')
df = pd.read_csv(csv_path)
print(f'{len(df):,} rows × {len(df.columns)} columns')
display(df.head(20))

## Spatial Trend Raster

**Theil–Sen GeoTIFF:** [Ate_Macoke_TheilSen_Trend_20261016_210357_745832.tif](<../results/SA20_satellite_maps/Ate_Macoke_NDVI_Trend_Analysis/Ate_Macoke_TheilSen_Trend_20261016_210357_745832.tif>)

Pixel-level slopes are expressed in NDVI units per year.


## Dissertation Results Interpretation

Landsat growing-season NDVI analysis from 2015 through 2023
identified spatially heterogeneous vegetation changes across
Ate_Macoke.

The descriptive spatial analysis classified 143.73 hectares
as declining, 107.37 hectares as relatively stable, and
7.92 hectares as increasing.

Parcel-level OLS regression indicated a negative slope
(−0.00684 NDVI/year), but the result was not statistically
significant (p = 0.3765).

These results document vegetation-greenness changes rather
than demonstrating their cause. Additional land-use,
precipitation, grazing-management, and field observations
would be needed to assess causal explanations.

**Data governance:** Oglala Lakota Nation — Tribal Data Sovereignty.


<!-- END ATE_MACOKE_MANAGED_GALLERY -->